In [1]:
from dotenv import load_dotenv
import os
# .env 파일을 불러와서 환경 변수로 설정
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
print(OPENAI_API_KEY[:10])

gsk_DICKFK


#### 1. CommaSeparatedListOutputParser

In [2]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
import csv

# 콤마로 구분된 리스트 출력 파서 초기화
output_parser = CommaSeparatedListOutputParser()

# 출력 형식 지침 가져오기
format_instructions = output_parser.get_format_instructions()
print(format_instructions)

Your response should be a list of comma separated values, eg: `foo, bar, baz` or `foo,bar,baz`


In [3]:

# 프롬프트 템플릿 설정
prompt = PromptTemplate(
    template="List five {subject}.\n{format_instructions}",
    input_variables=["subject"],
    partial_variables={"format_instructions": format_instructions},
)

# OpenAI 모델 설정
#model = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)
llm = ChatOpenAI(
    base_url="https://api.groq.com/openai/v1",  # Groq API 엔드포인트
    #model="meta-llama/llama-4-scout-17b-16e-instruct",  # Spring AI와 동일한 모델
    model="moonshotai/kimi-k2-instruct-0905",
    #model="openai/gpt-oss-120b",
    temperature=0.7
)
print(llm.model_name)


moonshotai/kimi-k2-instruct-0905


In [4]:

# 프롬프트, 모델, 출력 파서를 연결하여 체인 생성
chain = prompt | llm | output_parser

# "AI 관련 기술"에 대한 체인 호출 실행
result = chain.invoke({"subject": "AI 관련 기술"})

# 쉼표로 구분된 리스트 출력
print("AI 관련 기술 목록:")
print(result)

AI 관련 기술 목록:
['머신러닝', '딥러닝', '자연어처리', '컴퓨터비전', '강화학습']


In [5]:

# 결과 활용 예시: CSV 파일로 저장
csv_filename = "../data/ai_technologies.csv"
with open(csv_filename, "w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow(["AI 기술"])  # 헤더 추가
    for item in result:
        writer.writerow([item])

print(f" '{csv_filename}' 파일로 저장 완료!")


 '../data/ai_technologies.csv' 파일로 저장 완료!


#### 2. JsonOutputParser

In [6]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from langchain_openai import ChatOpenAI
import json

# JSON 출력 파서 초기화
parser = JsonOutputParser()

# 프롬프트 템플릿을 설정합니다.
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "당신은 과학 분야 전문가 AI입니다. 질문에 대해 체계적이고 간결한 답변을 JSON 형식으로 제공하세요."),
        ("user", "#Format: {format_instructions}\n\n#Question: {question}"),
    ]
)

# JSON 출력 형식 지침을 프롬프트에 적용
prompt = prompt.partial(format_instructions=parser.get_format_instructions())

# OpenAI 모델 설정
#llm = ChatOpenAI(model="gpt-3.5-turbo-0125", temperature=0)

# 프롬프트, 모델, 출력 파서를 연결하는 체인 생성
chain = prompt | llm | parser

# 질문 설정 (우주 탐사 관련 질문)
question = "최근 10년간 진행된 주요 우주 탐사 미션 3가지를 알려주세요. \
각 미션의 이름은 `mission_name`에, 목표는 `goal`에, 주관 기관은 `agency`에 담아 주세요."

# 체인 실행 및 JSON 응답 받기
response = chain.invoke({"question": question})
print(type(response))

# JSON 데이터 출력
print(json.dumps(response, indent=4, ensure_ascii=False))


<class 'list'>
[
    {
        "mission_name": "Perseverance (Mars 2020)",
        "goal": "화석 증거를 찾아 화성의 고대 미생물 생명跡 확인 및 시료 채취·보관",
        "agency": "NASA"
    },
    {
        "mission_name": "Hayabusa2",
        "goal": "근지 소행성 ‘류구’ 표면·내부 시료 채취 후 귀환",
        "agency": "JAXA"
    },
    {
        "mission_name": "James Webb Space Telescope (JWST)",
        "goal": "적외선 관측으로 초기 우주 형성·별·행성계 탄생 연구",
        "agency": "NASA·ESA·CSA"
    }
]


#### 3. PandasDataFrameOutputParser
* langchain 1.2 버전에서는 PandasDataFrameOutputParser 가 삭제 되었음

In [9]:
# import pandas as pd
# from langchain_core.output_parsers import PandasDataFrameOutputParser
# from langchain_core.prompts import PromptTemplate
# from langchain_openai import ChatOpenAI
# import re

# # Titanic 데이터셋 로드
# df = pd.read_csv('../data/titanic.csv')

# # ChatOpenAI 모델 초기화
# #llm = ChatOpenAI(temperature=0, model_name="gpt-3.5-turbo")

# # Pandas DataFrame Output Parser 설정
# parser = PandasDataFrameOutputParser(dataframe=df)

# # 형식 지침 출력
# format_instructions = parser.get_format_instructions()
# print("Format Instructions:\n", format_instructions)

# # 프롬프트 템플릿 설정
# prompt = PromptTemplate(
#     template=""" 
#     You are a helpful assistant that interacts with a Pandas DataFrame.
#     The DataFrame contains the following columns: {columns}.
    
#     Your task is to answer the user's query by generating a command in the following format:
#     {format_instructions}
    
#     User Query: {query}    
#     """,
#     input_variables=["query"],
#     partial_variables={
#         "format_instructions": format_instructions,
#         "columns": ", ".join(df.columns)
#     },
# )

# # 체인 생성
# chain = prompt | llm | parser

# # 모델 응답 받기
# try:
#     # **Name 열을 표시하십시오.**
#     print('Name 컬럼 출력')
#     df_query = "Show the Name column"

#     parser_output = chain.invoke({"query": df_query})
#     print(type(parser_output))
#     print(parser_output)

#     # **첫번째 행을 표시하십시오.**
#     print('첫번째 행 출력')
#     df_query2 = "Show first row"

#     parser_output2 = chain.invoke({"query": df_query2})
#     print(parser_output2)

#     #Please tell me the average value of the Fare column.
#     print('Fare 컬럼의 평균값 출력')
#     df_query3 = "Show me the average value of the Fare column."

#     parser_output3 = chain.invoke({"query": df_query3})
#     print(parser_output3)

# except Exception as e:
#     print(f"오류 발생: {e}")

# # 프롬프트 템플릿 설정
# prompt = PromptTemplate(
#     template=""" 
#     You are a helpful assistant that interacts with a Pandas DataFrame.
#     The DataFrame contains the following columns: {columns}.
    
#     Your task is to answer the user's query by generating a command in the following format:
#     {format_instructions}
    
#     User Query: {query}    
#     """,
#     input_variables=["query"],
#     partial_variables={
#         "format_instructions": format_instructions,
#         "columns": ", ".join(df.columns)
#     },
# )

# # 체인 생성
# chain = prompt | llm | parser

# # 모델 응답 받기
# try:
#     # **Name 열을 표시하십시오.**
#     print('Name 컬럼 출력')
#     df_query = "Show the Name column"

#     parser_output = chain.invoke({"query": df_query})
#     print(type(parser_output))
#     print(parser_output)

#     # **첫번째 행을 표시하십시오.**
#     print('첫번째 행 출력')
#     df_query2 = "Show first row"

#     parser_output2 = chain.invoke({"query": df_query2})
#     print(parser_output2)

#     #Please tell me the average value of the Fare column.
#     print('Fare 컬럼의 평균값 출력')
#     df_query3 = "Show me the average value of the Fare column."

#     parser_output3 = chain.invoke({"query": df_query3})
#     print(parser_output3)

# except Exception as e:
#     print(f"오류 발생: {e}")


#### PydanticOutputParser를 사용하여 다시 수정한 예제

In [13]:
import pandas as pd
import re
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

# Titanic 데이터셋 로드
df = pd.read_csv('../data/titanic.csv')

# Pydantic 모델 정의 (pandas 명령어 구조화)
class PandasCommand(BaseModel):
    """Pandas DataFrame에서 실행할 명령어"""
    command: str = Field(..., description="실행할 pandas 명령어 예: df['Name'], df.iloc[0], df['Fare'].mean()")

# ChatOpenAI 모델 초기화 (주석 해제 필요)
#llm = ChatOpenAI(temperature=0, model="gpt-3.5-turbo")
llm = ChatOpenAI(
    base_url="https://api.groq.com/openai/v1",  # Groq API 엔드포인트
    #model="meta-llama/llama-4-scout-17b-16e-instruct",  # Spring AI와 동일한 모델
    model="moonshotai/kimi-k2-instruct-0905",
    #model="openai/gpt-oss-120b",
    temperature=0.7
)

# PydanticOutputParser 설정
parser = PydanticOutputParser(pydantic_object=PandasCommand)

# 형식 지침 출력
format_instructions = parser.get_format_instructions()
print("Format Instructions:\n", format_instructions)
print("-" * 50)

# 프롬프트 템플릿 설정 (ChatPromptTemplate 사용 권장)
prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a pandas DataFrame expert for the Titanic dataset.
    
Columns: {columns}
    
User query에 맞는 **pandas 명령어만** 생성하세요. 다른 설명 없음."""),
    ("user", "{query}\n\n{format_instructions}")
])

# 입력 변수 부분 설정
prompt = prompt.partial(
    columns=", ".join(df.columns),
    format_instructions=format_instructions
)

# 체인 생성
chain = prompt | llm | parser

# 안전한 pandas 명령 실행 함수
def safe_execute_pandas(df, command: str):
    """허용된 pandas 명령어만 안전하게 실행"""
    safe_commands = {
        r"df\['[^']*'\]": lambda: eval(command, {"df": df}, {}),
        r'df\["[^"]*"\]': lambda: eval(command, {"df": df}, {}),
        r"df\.iloc\[\d+\]": lambda: eval(command, {"df": df}, {}),
        r"df\['[^']*'\]\.mean\(\)": lambda: eval(command, {"df": df}, {}),
        r"df\['[^']*'\]\.head\(\)": lambda: eval(command, {"df": df}, {}),
        r"df\.head\(\)": lambda: eval(command, {"df": df}, {}),
        r"df\.shape": lambda: eval(command, {"df": df}, {}),
    }
    
    command = command.strip()
    for pattern, executor in safe_commands.items():
        if re.match(pattern, command):
            try:
                return executor()
            except Exception as e:
                return f"실행 오류: {e}"
    return "허용되지 않은 명령어입니다."

# 테스트 실행
queries = [
    ("Name 컬럼 출력", "Show the Name column"),
    ("첫번째 행 출력", "Show first row"),
    ("Fare 컬럼 평균값 출력", "Show me the average value of the Fare column.")
]

for title, query in queries:
    print(f"\n{title}")
    print("=" * 40)
    
    try:
        result = chain.invoke({"query": query})
        print("생성된 명령어:", result.command)
        print("실행 결과:")
        output = safe_execute_pandas(df, result.command)
        print(output)
        print(type(output))
        
    except Exception as e:
        print(f"오류 발생: {e}")
        print(type(e))



Format Instructions:
 The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"description": "Pandas DataFrame에서 실행할 명령어", "properties": {"command": {"description": "실행할 pandas 명령어 예: df['Name'], df.iloc[0], df['Fare'].mean()", "title": "Command", "type": "string"}}, "required": ["command"]}
```
--------------------------------------------------

Name 컬럼 출력
생성된 명령어: df[['Name']].to_json(orient='records')
실행 결과:
허용되지 않은 명령어입니다.
<class 'str'>

첫번째 행 출력
생성된 명령어: df.iloc[0]
실행 결과:
PassengerId                          1
Survived                             0
Pclass                               3
Name 

In [ ]:
import pandas as pd
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
import os

# OpenAI API 키 설정 (환경변수에서 가져오기)
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")  # 또는 직접 설정

# Groq LLM 초기화 (최신 스타일)
llm = ChatOpenAI(
    api_key=OPENAI_API_KEY,
    base_url="https://api.groq.com/openai/v1",
    model="moonshotai/kimi-k2-instruct-0905",  # 또는 다른 모델
    temperature=0,
    seed=42,  # 핵심: Seed 고정으로 재현성 보장
)

# Pydantic 모델 정의 (테이블 데이터 구조화)
class TableData(BaseModel):
    """테이블 행을 나타내는 딕셔너리 리스트"""
    data: list[dict] = Field(..., description="테이블 행 데이터 (딕셔너리 리스트)")

# PydanticOutputParser 설정
parser = PydanticOutputParser(pydantic_object=TableData)

# ChatPromptTemplate 설정 (최신 LCEL 스타일)
prompt = ChatPromptTemplate.from_messages([
    ("system", """당신은 테이블 형태의 데이터를 생성하는 AI 어시스턴트입니다.
    매번 **동일한 데이터**를 정확히 동일한 순서로 반환하세요.
    반드시 다음 JSON 스키마를 정확히 따라야 합니다:
    {format_instructions}
    
    데이터는 현실적이고 일관성 있게 생성하세요."""),
    ("user", "{query}")
])

# 부분 변수 설정
prompt = prompt.partial(format_instructions=parser.get_format_instructions())

# 체인 생성
chain = prompt | llm | parser

# DataFrame 생성 함수
def generate_dataframe(user_query: str) -> pd.DataFrame:
    try:
        print(" 쿼리 처리 중:", user_query[:50] + "...")
        
        # 모델 호출
        result = chain.invoke({"query": user_query})
        print(" 생성된 JSON 데이터:", result)
        
        # DataFrame으로 변환
        df = pd.DataFrame(result.data)
        
        print("\n 생성된 DataFrame:")
        print(df)
        print(f"\n 데이터 형태: {df.shape} ({len(df.columns)} columns)")
        return df
        
    except Exception as e:
        print(f" 오류 발생: {e}")
        return pd.DataFrame()


In [25]:

# 실행 테스트
print("2024년 상반기 서울 아파트 평균 매매 가격 데이터 생성")
print("=" * 60)

df_seoul_housing = generate_dataframe(
    "2024년 상반기 서울 아파트 평균 매매가격 데이터셋을 생성해주세요. "
    "컬럼: 구(District), 평균가격_억원(Average_Price_100M_KRW), 거래건수(Transactions), "
    "전년동기대비증감률_퍼센트(YoY_Change_Percent). "
    "서울의 25개 구에 대한 현실적인 데이터를 생성해주세요."
)

print("\n" + "=" * 60)
print("최종 결과:")
print(df_seoul_housing)

2024년 상반기 서울 아파트 평균 매매 가격 데이터 생성
 쿼리 처리 중: 2024년 상반기 서울 아파트 평균 매매가격 데이터셋을 생성해주세요. 컬럼: 구(Distr...
 생성된 JSON 데이터: data=[{'District': '강남구', 'Average_Price_100M_KRW': 35.2, 'Transactions': 1284, 'YoY_Change_Percent': -3.1}, {'District': '강동구', 'Average_Price_100M_KRW': 16.8, 'Transactions': 892, 'YoY_Change_Percent': -1.2}, {'District': '강북구', 'Average_Price_100M_KRW': 9.3, 'Transactions': 654, 'YoY_Change_Percent': -0.8}, {'District': '강서구', 'Average_Price_100M_KRW': 14.7, 'Transactions': 1051, 'YoY_Change_Percent': 0.0}, {'District': '관악구', 'Average_Price_100M_KRW': 11.5, 'Transactions': 743, 'YoY_Change_Percent': -0.2}, {'District': '광진구', 'Average_Price_100M_KRW': 13.0, 'Transactions': 627, 'YoY_Change_Percent': 0.7}, {'District': '구로구', 'Average_Price_100M_KRW': 10.9, 'Transactions': 798, 'YoY_Change_Percent': -1.5}, {'District': '금천구', 'Average_Price_100M_KRW': 9.8, 'Transactions': 512, 'YoY_Change_Percent': -2.1}, {'District': '노원구', 'Average_Price_100M_KRW': 9.9, 'Transactions': 8

In [ ]:
df_seoul_housing

In [26]:
print('2024년 서울 지하철역별 유동 인구 데이터')
# [예제 2] 2024년 서울 지하철역별 유동 인구 데이터
df_seoul_subway = generate_dataframe(
    #"Generate a dataset of the top 10 busiest subway stations in Seoul in 2024 with columns: Station Name, Line Number, Daily Passenger Volume, and Weekday vs Weekend Ratio."
    "2024년 서울 지하철 승객 이용량이 가장 많은 상위 10개 역의 데이터셋을 생성해주세요. \
        컬럼은 다음과 같습니다: 역명(Station_Name), 호선(Line_Number),\
              일평균승객수_만명(Daily_Passengers_10K), 평일대주말비율(Weekday_Weekend_Ratio).\
                  실제 서울 지하철의 주요 역들을 기반으로 현실적인 데이터를 생성해주세요."
)

#if df_seoul_subway is not None:
df_seoul_subway.head()


2024년 서울 지하철역별 유동 인구 데이터
 쿼리 처리 중: 2024년 서울 지하철 승객 이용량이 가장 많은 상위 10개 역의 데이터셋을 생성해주세요....
 생성된 JSON 데이터: data=[{'Station_Name': '서울역', 'Line_Number': 1, 'Daily_Passengers_10K': 30.2, 'Weekday_Weekend_Ratio': '1.6:1'}, {'Station_Name': '강남', 'Line_Number': 2, 'Daily_Passengers_10K': 28.5, 'Weekday_Weekend_Ratio': '1.7:1'}, {'Station_Name': '신도림', 'Line_Number': 2, 'Daily_Passengers_10K': 25.9, 'Weekday_Weekend_Ratio': '1.9:1'}, {'Station_Name': '사당', 'Line_Number': 2, 'Daily_Passengers_10K': 24.3, 'Weekday_Weekend_Ratio': '1.4:1'}, {'Station_Name': '왕십리', 'Line_Number': 2, 'Daily_Passengers_10K': 23.8, 'Weekday_Weekend_Ratio': '1.6:1'}, {'Station_Name': '종로3가', 'Line_Number': 1, 'Daily_Passengers_10K': 22.0, 'Weekday_Weekend_Ratio': '1.8:1'}, {'Station_Name': '동대문역사문화공원', 'Line_Number': 2, 'Daily_Passengers_10K': 21.6, 'Weekday_Weekend_Ratio': '1.5:1'}, {'Station_Name': '합정', 'Line_Number': 2, 'Daily_Passengers_10K': 20.5, 'Weekday_Weekend_Ratio': '1.7:1'}, {'Station_Name': '신림', 'Line_N

,Station_Name,Line_Number,Daily_Passengers_10K,Weekday_Weekend_Ratio
0,서울역,1,30.2,1.6:1
1,강남,2,28.5,1.7:1
2,신도림,2,25.9,1.9:1
3,사당,2,24.3,1.4:1
4,왕십리,2,23.8,1.6:1


In [27]:
df_seoul_subway

,Station_Name,Line_Number,Daily_Passengers_10K,Weekday_Weekend_Ratio
0,서울역,1,30.2,1.6:1
1,강남,2,28.5,1.7:1
2,신도림,2,25.9,1.9:1
3,사당,2,24.3,1.4:1
4,왕십리,2,23.8,1.6:1
5,종로3가,1,22.0,1.8:1
6,동대문역사문화공원,2,21.6,1.5:1
7,합정,2,20.5,1.7:1
8,신림,2,20.1,1.3:1
9,건대입구,2,19.7,1.6:1


In [29]:
print('한국 5대 편의점 브랜드별 2024년 매출 및 점포 수')
# [예제 3] 한국 5대 편의점 브랜드별 2024년 매출 및 점포 수
df_korean_convenience_stores = generate_dataframe(
    #"Create a dataset of the top 5 convenience store brands in Korea in 2024 with columns: Brand Name, Number of Stores, Total Revenue (in billion KRW), and Market Share (%)."
    "2024년 한국의 편의점 브랜드 상위 5개사 데이터셋을 생성해주세요.\
          컬럼은 다음과 같습니다: 브랜드명(Brand_Name), 점포수(Store_Count), \
            총매출_조원(Revenue_Trillion_KRW), 시장점유율_퍼센트(Market_Share_Percent).\
                  CU, GS25, 세븐일레븐, 이마트24, 미니스톱 등 실제 한국 편의점 브랜드를 기반으로 현실적인 데이터를 생성해주세요."
)
df_korean_convenience_stores

한국 5대 편의점 브랜드별 2024년 매출 및 점포 수
 쿼리 처리 중: 2024년 한국의 편의점 브랜드 상위 5개사 데이터셋을 생성해주세요.          컬럼...
 생성된 JSON 데이터: data=[{'Brand_Name': 'CU', 'Store_Count': 15234, 'Revenue_Trillion_KRW': 15.7, 'Market_Share_Percent': 33.8}, {'Brand_Name': 'GS25', 'Store_Count': 13827, 'Revenue_Trillion_KRW': 14.3, 'Market_Share_Percent': 30.9}, {'Brand_Name': '세븐일레븐', 'Store_Count': 9785, 'Revenue_Trillion_KRW': 8.2, 'Market_Share_Percent': 17.7}, {'Brand_Name': '이마트24', 'Store_Count': 5135, 'Revenue_Trillion_KRW': 5.1, 'Market_Share_Percent': 11.0}, {'Brand_Name': '미니스톱', 'Store_Count': 2335, 'Revenue_Trillion_KRW': 2.1, 'Market_Share_Percent': 4.5}]

 생성된 DataFrame:
  Brand_Name  Store_Count  Revenue_Trillion_KRW  Market_Share_Percent
0         CU        15234                  15.7                  33.8
1       GS25        13827                  14.3                  30.9
2      세븐일레븐         9785                   8.2                  17.7
3      이마트24         5135                   5.1                 

,Brand_Name,Store_Count,Revenue_Trillion_KRW,Market_Share_Percent
0,CU,15234,15.7,33.8
1,GS25,13827,14.3,30.9
2,세븐일레븐,9785,8.2,17.7
3,이마트24,5135,5.1,11.0
4,미니스톱,2335,2.1,4.5
